## Cell 1: Load the Dataset

Import the libraries, read the CSV file and find the pollutant, location and weather columns.

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

file_path = "air_quality.csv"   # <-- change this to your dataset file

try:
    data = pd.read_csv(file_path)
    data.columns = data.columns.str.strip()      # remove extra spaces from column names
    print("Dataset loaded successfully!")
    print("Rows:", data.shape[0], "| Columns:", data.shape[1])
except Exception as error:
    print("Error reading the file:", error)

else:
    # Find the important columns automatically (edit the patterns if your names differ)
    pollutant_pattern = r"^(pm_?2\.?5|pm_?10|no2|so2|co|o3|nox|nh3|aqi)($|[^a-z])"

    available_pollutants = [
        col for col in data.columns
        if re.match(pollutant_pattern, col.lower())
        and pd.api.types.is_numeric_dtype(data[col])
    ]

    location_col = None
    for col in data.columns:
        if any(word in col.lower() for word in ["city", "location", "station", "area", "region"]):
            location_col = col
            break

    weather_columns = [
        col for col in data.columns
        if any(word in col.lower() for word in ["temperature", "humidity", "wind", "pressure"])
        and pd.api.types.is_numeric_dtype(data[col])
    ]

    print("\nPollutant columns:", available_pollutants)
    print("Location column  :", location_col)
    print("Weather columns  :", weather_columns)

## Cell 2: Explore the Dataset

Look at the first rows, shape, data types, summary statistics and missing values.

In [ ]:
print("First 5 rows of the data:")
display(data.head())

print("\nNumber of rows and columns:", data.shape)

print("\nColumns and data types:")
print(data.dtypes)

print("\nSummary statistics:")
display(data.describe())

print("\nMissing values in each column:")
print(data.isnull().sum())

## Cell 3: Clean the Dataset

Handle missing values, remove duplicate rows and prepare the data for analysis.

In [ ]:
numeric_cols = data.select_dtypes(include="number").columns
text_cols = data.select_dtypes(exclude="number").columns

# Step 1: show the missing values
print("Missing values in each column:")
missing = data.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

# Step 2: compare four ways of handling them
print("\nComparing ways to handle missing values:")

dropped = data.dropna()
print("1. Drop rows          -> rows left:", dropped.shape[0], "of", data.shape[0])

mean_filled = data.copy()
mean_filled[numeric_cols] = mean_filled[numeric_cols].fillna(mean_filled[numeric_cols].mean())
print("2. Fill with mean     -> missing numeric values left:", mean_filled[numeric_cols].isnull().sum().sum())

median_filled = data.copy()
median_filled[numeric_cols] = median_filled[numeric_cols].fillna(median_filled[numeric_cols].median())
print("3. Fill with median   -> missing numeric values left:", median_filled[numeric_cols].isnull().sum().sum())

mode_filled = data.copy()
for col in data.columns:
    if data[col].isnull().any() and not data[col].mode().empty:
        mode_filled[col] = mode_filled[col].fillna(data[col].mode()[0])
print("4. Fill with mode     -> missing values left:", mode_filled.isnull().sum().sum())

# Step 3: remove duplicate rows
before = len(data)
data = data.drop_duplicates()
print("\nDuplicate rows removed:", before - len(data))

# Step 4: final cleaning - numeric columns get the mean, text columns get the mode
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].mean())
for col in text_cols:
    if data[col].isnull().any() and not data[col].mode().empty:
        data[col] = data[col].fillna(data[col].mode()[0])

print("\nMissing values after cleaning:", data.isnull().sum().sum())
print("Final shape:", data.shape)

## Cell 4: Analyze the Dataset

Find the most and least polluted locations.

In [ ]:
if location_col and available_pollutants:

    location_pollution = data.groupby(location_col)[available_pollutants].mean()

    location_pollution["Average Pollution"] = (
        location_pollution[available_pollutants].mean(axis=1)
    )

    location_pollution = location_pollution.sort_values(
        by="Average Pollution",
        ascending=False
    )

    # show 5 locations, or fewer if the dataset has only a few locations
    top_n = min(5, len(location_pollution) // 2)

    print(f"Top {top_n} most polluted locations:")
    display(location_pollution.head(top_n))

    print(f"\nTop {top_n} least polluted locations:")
    display(location_pollution.tail(top_n).sort_values(by="Average Pollution"))

    print("\nMost Polluted Location:", location_pollution["Average Pollution"].idxmax())
    print("Least Polluted Location:", location_pollution["Average Pollution"].idxmin())

    # Highest and lowest locations for each pollutant
    for col in available_pollutants:
        print("\n--- " + col + " ---")
        print(f"{top_n} highest locations:")
        print(location_pollution[col].nlargest(top_n).round(2))
        print(f"{top_n} lowest locations:")
        print(location_pollution[col].nsmallest(top_n).round(2))

else:
    print("Location or pollutant columns not found.")

## Cell 5: Correlation Analysis

Check how pollution is related to temperature, humidity and wind speed.

In [ ]:
if weather_columns and available_pollutants:

    correlation = data[available_pollutants + weather_columns].corr()
    weather_pollution = correlation.loc[available_pollutants, weather_columns]

    print("Correlation between pollution and weather:\n")
    for pollutant in available_pollutants:
        for weather in weather_columns:
            print(f"{pollutant} and {weather}: {weather_pollution.loc[pollutant, weather]:.2f}")

    display(weather_pollution)

    strongest = weather_pollution.abs().stack().idxmax()
    print("\nStrongest relationship:", strongest[0], "and", strongest[1],
          "->", round(weather_pollution.loc[strongest], 2))

else:
    print("Weather or pollutant columns not found.")

## Cell 6: Visualize the Dataset

Show the results as bar charts, scatter plots and a heatmap.

In [ ]:

# 1. Bar charts: most and least polluted locations
if location_col and available_pollutants:

    chart_n = min(10, len(location_pollution) // 2)
    top10 = location_pollution["Average Pollution"].head(chart_n)
    bottom10 = location_pollution["Average Pollution"].tail(chart_n).sort_values()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.barplot(x=top10.values, y=top10.index, hue=top10.index,
                palette="Reds_r", legend=False, ax=axes[0])
    axes[0].set_title(f"{chart_n} Most Polluted Locations")
    axes[0].set_xlabel("Average Pollution")
    axes[0].set_ylabel(location_col)

    sns.barplot(x=bottom10.values, y=bottom10.index, hue=bottom10.index,
                palette="Greens", legend=False, ax=axes[1])
    axes[1].set_title(f"{chart_n} Least Polluted Locations")
    axes[1].set_xlabel("Average Pollution")
    axes[1].set_ylabel("")

    plt.tight_layout()
    plt.show()

# 2. Scatter plots: average pollution against each weather factor
if weather_columns and available_pollutants:

    row_pollution = data[available_pollutants].mean(axis=1)

    fig, axes = plt.subplots(1, len(weather_columns),
                             figsize=(5 * len(weather_columns), 4), squeeze=False)

    for ax, weather in zip(axes[0], weather_columns):
        sns.regplot(x=data[weather], y=row_pollution, ax=ax,
                    scatter_kws={"alpha": 0.4, "s": 15}, line_kws={"color": "red"})
        ax.set_title("Pollution vs " + weather)
        ax.set_xlabel(weather)
        ax.set_ylabel("Average Pollution")

    plt.tight_layout()
    plt.show()

    # 3. Heatmap: pollution and weather relationship
    plt.figure(figsize=(10, 6))
    sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Pollution and Weather Relationship")
    plt.show()

else:
    print("Weather or pollutant columns not found.")